In [1]:
%%capture
# Gerekli kütüphaneleri yükle
!pip install chromadb sentence-transformers tqdm
!pip install --upgrade unsloth
!pip install --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [7]:
import os
import re
import json
import torch
from threading import Thread
from typing import List, Dict, Optional
import gradio as gr

# ChromaDB imports
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
import numpy as np

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# Unsloth imports
from unsloth import FastLanguageModel
from transformers import TextIteratorStreamer

In [3]:
# ============================================================================
# CONFIGURATION
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

CHROMADB_PERSIST_DIR = "/content/drive/MyDrive/HukukPusulasi/legal_chroma_db"
CHROMADB_COLLECTION_NAME = "legal_documents_v2"
EMBEDDING_MODEL_NAME = "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr"

# HUGGING FACE'DEN MODELİ YÜKLE
HF_MODEL_ID = "beyzasn/hukuk-pusulasi-llm-v1"
MAX_SEQ_LENGTH = 8192

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================================
# ✅ DÜZELTİLMİŞ DISTANCE ANALYZER
# ============================================================================

class DistanceAnalyzer:
    """
    ✅ DOĞRU THRESHOLD'LAR - ChromaDB L2 distance için optimize edilmiş
    """

    # GERÇEKÇİ THRESHOLD'LAR (ChromaDB L2 squared distance için)
    QUALITY_THRESHOLDS = {
        'excellent': 200.0,   # 0-200 arası -> mükemmel
        'good': 350.0,        # 200-350 arası -> iyi
        'acceptable': 500.0,  # 350-500 arası -> kabul edilebilir
        'poor': 650.0         # 500-650 arası -> zayıf
    }

    def __init__(self):
        self.distance_history = []

    def categorize_distance(self, distance: float) -> str:
        """Mesafeyi kategorize et"""
        if distance < self.QUALITY_THRESHOLDS['excellent']:
            return 'excellent'
        elif distance < self.QUALITY_THRESHOLDS['good']:
            return 'good'
        elif distance < self.QUALITY_THRESHOLDS['acceptable']:
            return 'acceptable'
        elif distance < self.QUALITY_THRESHOLDS['poor']:
            return 'poor'
        else:
            return 'very_poor'

    def filter_by_quality(self, results: Dict, quality_level: str = 'acceptable') -> Dict:
        """
        ✅ DÜZELTME: Gerçekçi threshold'larla filtreleme
        """
        if not results.get('documents') or not results['documents']:
            return {'documents': [], 'metadatas': [], 'distances': [], 'n_results': 0}

        docs = results['documents']
        metas = results['metadatas']
        dists = results['distances']

        threshold = self.QUALITY_THRESHOLDS.get(quality_level, 500.0)

        filtered_docs = []
        filtered_metas = []
        filtered_dists = []

        for doc, meta, dist in zip(docs, metas, dists):
            if dist <= threshold:
                filtered_docs.append(doc)
                filtered_metas.append(meta)
                filtered_dists.append(dist)

        # Mesafe geçmişine ekle
        if filtered_dists:
            self.distance_history.extend(filtered_dists)

        return {
            'documents': filtered_docs,
            'metadatas': filtered_metas,
            'distances': filtered_dists,
            'n_results': len(filtered_docs)
        }

    def analyze_results(self, distances: List[float]) -> Dict:
        """Sonuç kalitesini analiz et"""
        if not distances:
            return {'status': 'no_results', 'quality': None}

        categories = {k: 0 for k in ['excellent', 'good', 'acceptable', 'poor', 'very_poor']}

        for dist in distances:
            cat = self.categorize_distance(dist)
            categories[cat] += 1

        avg_distance = sum(distances) / len(distances)
        min_distance = min(distances)

        # Genel kalite değerlendirmesi
        if categories['excellent'] >= 2:
            overall_quality = 'excellent'
        elif categories['good'] >= 3:
            overall_quality = 'good'
        elif categories['acceptable'] >= 2:
            overall_quality = 'acceptable'
        else:
            overall_quality = 'poor'

        return {
            'status': 'success',
            'quality': overall_quality,
            'categories': categories,
            'avg_distance': avg_distance,
            'min_distance': min_distance,
            'n_results': len(distances)
        }

# ============================================================================
# SEMANTIC QUERY PROCESSOR
# ============================================================================

class SemanticQueryProcessor:
    """
    Semantic similarity ile query processing
    """

    def __init__(self, embedding_model_name: str):
        self.embedder = SentenceTransformer(embedding_model_name)

        # Yasal konsept örnekleri
        self.legal_concept_examples = {
            'cayma_hakki': [
                "ürünü iade etmek istiyorum",
                "siparişimi iptal edebilir miyim",
                "aklımı değiştirdim geri verebilir miyim",
                "cayma hakkım var mı",
                "vazgeçme süresi ne kadar"
            ],
            'ayipli_mal': [
                "ürün bozuk geldi",
                "kusurlu mal aldım",
                "telefon kırık çıktı",
                "defolu ürün ne yapmalıyım",
                "çalışmayan cihaz iade"
            ],
            'garanti': [
                "garanti süresi ne kadar",
                "arıza için garanti var mı",
                "üretici garantisi",
                "tamire gönderebilir miyim"
            ],
            'tazminat': [
                "zarar tazminatı isteyebilir miyim",
                "maddi zarar",
                "manevi tazminat",
                "zararımı karşılayabilir miyim"
            ]
        }

        # Sözleşme tipi örnekleri
        self.contract_type_examples = {
            'mesafeli': [
                "online alışveriş",
                "internet üzerinden aldım",
                "e-ticaret sitesinden",
                "web sitesinden sipariş",
                "uzaktan satış"
            ],
            'kapidan': [
                "kapıda satış",
                "eve gelen satıcı",
                "kapımda sattılar",
                "iş yeri dışında satış"
            ],
            'taksitli': [
                "taksitle aldım",
                "kredi kartı taksit",
                "ödeme planı"
            ],
            'konut': [
                "ev aldım",
                "daire satın aldım",
                "ön ödemeli konut",
                "konut finansmanı"
            ]
        }

        # Embedding'leri önceden hesapla
        self.legal_embeddings = self._prepare_concept_embeddings(self.legal_concept_examples)
        self.contract_embeddings = self._prepare_concept_embeddings(self.contract_type_examples)

    def _prepare_concept_embeddings(self, examples_dict: Dict) -> Dict:
        """Her konsept için embedding'leri hazırla"""
        result = {}
        for concept, examples in examples_dict.items():
            embeddings = self.embedder.encode(examples, convert_to_tensor=False)
            result[concept] = np.mean(embeddings, axis=0)
        return result

    def _semantic_similarity(self, query: str, concept_embeddings: Dict, threshold: float = 0.5) -> List[tuple]:
        """Sorgu ile konseptler arasında semantic similarity hesapla"""
        query_embedding = self.embedder.encode([query], convert_to_tensor=False)[0]

        similarities = []
        for concept, concept_emb in concept_embeddings.items():
            sim = np.dot(query_embedding, concept_emb) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(concept_emb)
            )
            if sim > threshold:
                similarities.append((concept, float(sim)))

        return sorted(similarities, key=lambda x: x[1], reverse=True)

    def detect_contract_type(self, query: str) -> Optional[str]:
        """Semantic similarity ile sözleşme tipini tespit et"""
        matches = self._semantic_similarity(query, self.contract_embeddings, threshold=0.45)
        return matches[0][0] if matches else None

    def extract_legal_concepts(self, query: str) -> List[tuple]:
        """Semantic similarity ile yasal konseptleri çıkar"""
        return self._semantic_similarity(query, self.legal_embeddings, threshold=0.40)

    def enrich_query(self, query: str) -> Dict:
        """Sorguyu semantic olarak zenginleştir"""
        contract_type = self.detect_contract_type(query)
        legal_concepts = self.extract_legal_concepts(query)

        enriched_parts = [query]

        # Sözleşme tipi ekleme
        if contract_type:
            type_map = {
                'mesafeli': 'mesafeli sözleşme internet alışverişi',
                'kapidan': 'kapıdan satış doğrudan satış',
                'taksitli': 'taksitle satış kredi',
                'konut': 'ön ödemeli konut finansmanı'
            }
            enriched_parts.append(type_map.get(contract_type, ''))

        # Yasal konsept ekleme
        concept_map = {
            'cayma_hakki': 'cayma hakkı iade vazgeçme',
            'ayipli_mal': 'ayıplı mal kusurlu ürün defolu',
            'garanti': 'garanti satıcı garantisi üretici garantisi',
            'tazminat': 'tazminat zarar ziyan'
        }

        for concept, score in legal_concepts[:2]:
            if score > 0.5:
                enriched_parts.append(concept_map.get(concept, ''))

        # Metadata filtresi
        metadata_filters = None
        if contract_type:
            if contract_type == 'mesafeli':
                metadata_filters = {
                    "$or": [
                        {"file_name": {"$eq": "Regulation_MESAFELI_SOZLESMELER_YONETMELIGI.pdf"}},
                        {"file_name": {"$eq": "Law_TUKETICININ_KORUNMASI_HAKKINDA_KANUN.pdf"}},
                    ]
                }
            elif contract_type == 'konut':
                metadata_filters = {"file_name": {"$eq": "Regulation_KONUT_FINANSMANI_SOZLESMELERI_YONETMELIGI.pdf"}}
            elif contract_type == 'kapidan':
                metadata_filters = {"file_name": {"$eq": "Regulation_DOGRUDAN_SATISLAR_HAKKINDA_YONETMELIK.pdf"}}

        return {
            'original': query,
            'enriched': ' '.join(filter(None, enriched_parts)),
            'contract_type': contract_type,
            'legal_concepts': [c for c, s in legal_concepts],
            'concept_scores': legal_concepts,
            'metadata_filters': metadata_filters,
        }

# ============================================================================
# ADAPTIVE SEARCH STRATEGY
# ============================================================================

class AdaptiveSearchStrategy:
    """Sorgu tipine göre adaptif arama stratejisi"""

    def __init__(self, vector_store):
        self.vector_store = vector_store

    def determine_search_strategy(self, query_analysis: Dict) -> Dict:
        """Sorgu analizine göre en iyi arama stratejisini belirle"""

        has_strong_concept = any(score > 0.6 for _, score in query_analysis.get('concept_scores', []))
        has_clear_contract = query_analysis.get('contract_type') is not None

        strategy = {
            'semantic_weight': 0.5,
            'filtered_weight': 0.3,
            'hybrid_weight': 0.2,
            'n_results_semantic': 10,
            'n_results_filtered': 5,
            'quality_threshold': 'good'  # ✅ DÜZELTME: 'acceptable' yerine 'good'
        }

        if has_strong_concept and has_clear_contract:
            strategy.update({
                'semantic_weight': 0.4,
                'filtered_weight': 0.5,
                'n_results_semantic': 8,
                'n_results_filtered': 7,
            })
        elif has_strong_concept:
            strategy.update({
                'semantic_weight': 0.7,
                'filtered_weight': 0.1,
                'n_results_semantic': 15,
                'quality_threshold': 'acceptable'
            })
        elif has_clear_contract:
            strategy.update({
                'semantic_weight': 0.3,
                'filtered_weight': 0.6,
                'n_results_filtered': 10,
            })
        else:
            strategy.update({
                'semantic_weight': 0.6,
                'n_results_semantic': 20,
                'quality_threshold': 'acceptable'
            })

        return strategy

# ============================================================================
# ✅ DÜZELTİLMİŞ VECTOR STORE
# ============================================================================

class SmartVectorStore:
    """
    ✅ DÜZELTME: Doğru DistanceAnalyzer ile çalışan vector store
    """

    def __init__(self, persist_dir: str, collection_name: str, model_name: str):
        self.persist_directory = persist_dir
        os.makedirs(persist_dir, exist_ok=True)

        self.client = chromadb.PersistentClient(
            path=persist_dir,
            settings=Settings(anonymized_telemetry=False, allow_reset=True)
        )

        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=model_name, device=DEVICE
        )

        try:
            self.collection = self.client.get_collection(
                name=collection_name,
                embedding_function=self.embedding_function
            )
            print(f"✅ ChromaDB: {self.collection.count()} doküman yüklendi")
        except Exception as e:
            print(f"❌ ChromaDB collection bulunamadı: {e}")
            self.collection = None

        # ✅ DÜZELTME: Doğru analyzer'ı kullan
        self.semantic_processor = SemanticQueryProcessor(model_name)
        self.search_strategy = AdaptiveSearchStrategy(self)
        self.distance_analyzer = DistanceAnalyzer()  # ✅ Yeni analyzer

    def smart_search(self, query: str, n_results: int = 5) -> Dict:
        """✅ DÜZELTME: Gerçekçi threshold'larla arama"""

        # 1. Semantic analiz
        analysis = self.semantic_processor.enrich_query(query)

        print(f"\n🔍 Sorgu: '{query}'")
        print(f"📊 Semantic Analiz:")
        print(f"   • Sözleşme tipi: {analysis['contract_type'] or 'Belirsiz'}")
        print(f"   • Yasal konseptler: {[f'{c} ({s:.2f})' for c, s in analysis.get('concept_scores', [])]}")
        print(f"🎯 Zenginleştirilmiş: {analysis['enriched'][:100]}...")

        # 2. Arama stratejisi
        strategy = self.search_strategy.determine_search_strategy(analysis)
        print(f"\n📋 Strateji: Semantic={strategy['semantic_weight']:.1f}, Filtered={strategy['filtered_weight']:.1f}")

        all_results = []

        # 3. Filtered search
        if analysis['metadata_filters']:
            print("\n1️⃣ Filtered search...")
            try:
                filtered = self.collection.query(
                    query_texts=[analysis['enriched']],
                    n_results=strategy['n_results_filtered'],
                    where=analysis['metadata_filters'],
                    include=["documents", "metadatas", "distances"]
                )

                if filtered['ids'][0]:
                    for doc, meta, dist in zip(
                        filtered['documents'][0],
                        filtered['metadatas'][0],
                        filtered['distances'][0]
                    ):
                        all_results.append({
                            'document': doc,
                            'metadata': meta,
                            'distance': dist,
                            'strategy': 'filtered',
                            'score': dist * (1 - strategy['filtered_weight'])
                        })
                    print(f"   ✅ {len(filtered['ids'][0])} sonuç")
            except Exception as e:
                print(f"   ⚠️ Filtre hatası: {e}")

        # 4. Semantic search
        print("\n2️⃣ Semantic search...")
        semantic = self.collection.query(
            query_texts=[analysis['enriched']],
            n_results=strategy['n_results_semantic'],
            include=["documents", "metadatas", "distances"]
        )

        for doc, meta, dist in zip(
            semantic['documents'][0],
            semantic['metadatas'][0],
            semantic['distances'][0]
        ):
            all_results.append({
                'document': doc,
                'metadata': meta,
                'distance': dist,
                'strategy': 'semantic',
                'score': dist * (1 - strategy['semantic_weight'])
            })

        print(f"   ✅ {len(semantic['ids'][0])} sonuç")

        # 5. Deduplicate & sort
        seen_ids = set()
        unique_results = []

        for r in sorted(all_results, key=lambda x: x['score']):
            doc_hash = hash(r['document'][:100])
            if doc_hash not in seen_ids:
                seen_ids.add(doc_hash)
                unique_results.append(r)

        # 6. ✅ DÜZELTME: Gerçekçi threshold'larla filtreleme
        final_results = self.distance_analyzer.filter_by_quality(
            {
                'documents': [r['document'] for r in unique_results],
                'metadatas': [r['metadata'] for r in unique_results],
                'distances': [r['distance'] for r in unique_results],
            },
            quality_level=strategy['quality_threshold']
        )

        print(f"\n✅ Toplam {final_results['n_results']} kaliteli sonuç")
        if final_results['distances']:
            print(f"📊 Mesafeler: {[f'{d:.1f}' for d in final_results['distances'][:3]]}")

        return {
            'query': query,
            'n_results': min(final_results['n_results'], n_results),
            'documents': final_results['documents'][:n_results],
            'metadatas': final_results['metadatas'][:n_results],
            'distances': final_results['distances'][:n_results],
            'analysis': analysis
        }

# ============================================================================
# SOURCE FORMATTER
# ============================================================================

def format_source(doc: str, meta: Dict) -> str:
    """Content'ten ve metadata'dan kaynak bilgisini akıllıca çıkarır"""

    doc_type = meta.get('doc_type', '')
    content_lines = doc.strip().split('\n')

    # Mahkeme kararları için
    if doc_type == 'court_decision':
        for i, line in enumerate(content_lines[:3]):
            if line.startswith('T.C.'):
                mahkeme_adi = line.replace('T.C.', '').strip()

                esas_no = None
                for check_line in content_lines[:5]:
                    if 'ESAS NO' in check_line or 'Esas' in check_line:
                        esas_match = re.search(r'(\d{4}/\d+)', check_line)
                        if esas_match:
                            esas_no = esas_match.group(1)
                            break

                if mahkeme_adi and esas_no:
                    return f"{mahkeme_adi} - {esas_no} Esas"
                elif mahkeme_adi:
                    return mahkeme_adi

    # Yönetmelik/Kanun için
    elif doc_type in ['regulation', 'law']:
        if content_lines:
            baslik = content_lines[0].strip()

            if 'YÖNETMELİK' in baslik or 'KANUN' in baslik:
                article_num = meta.get('article_number')
                if article_num:
                    return f"{baslik} - Madde {article_num}"
                return baslik

    # Fallback
    file_name = meta.get('file_name', 'Kaynak')

    if file_name.startswith('Regulation_'):
        clean_name = file_name.replace('Regulation_', '').replace('.pdf', '').replace('_', ' ')
        article_num = meta.get('article_number')
        if article_num:
            return f"{clean_name} - Madde {article_num}"
        return clean_name
    elif file_name.startswith('Law_'):
        clean_name = file_name.replace('Law_', '').replace('.pdf', '').replace('_', ' ')
        article_num = meta.get('article_number')
        if article_num:
            return f"{clean_name} - Madde {article_num}"
        return clean_name

    return file_name.replace('.pdf', '').replace('_', ' ')

# ============================================================================
# SMART PROMPT ENGINEERING
# ============================================================================

def create_smart_prompt(query: str, search_results: Dict) -> tuple:
    """Semantic analiz sonuçlarına göre optimized prompt"""

    analysis = search_results.get('analysis', {})

    # Context hazırlama
    contexts = []
    source_details = []

    for i, (doc, meta) in enumerate(zip(
        search_results['documents'],
        search_results['metadatas']
    )):
        source_info = format_source(doc, meta)
        doc_text = doc.strip()[:600] + "..." if len(doc.strip()) > 600 else doc.strip()
        contexts.append(f"[Kaynak {i+1}]:\n{doc_text}")
        source_details.append(f"[{i+1}] {source_info}")

    context_block = "\n\n".join(contexts)

    # Sözleşme tipine göre özel talimat
    contract_guidance = ""
    if analysis.get('contract_type') == 'mesafeli':
        contract_guidance = "\n• Bu mesafeli sözleşme sorusu - cayma hakkı ve iade süreçlerine odaklan"
    elif 'ayipli_mal' in analysis.get('legal_concepts', []):
        contract_guidance = "\n• Bu ayıplı mal sorusu - tüketici haklarını ve çözüm yollarını açıkla"

    # Konseptlere göre ek talimat
    concept_guidance = ""
    concepts = analysis.get('legal_concepts', [])
    if 'cayma_hakki' in concepts:
        concept_guidance = "\n• Cayma hakkı sürelerini ve koşullarını mutlaka belirt"
    elif 'ayipli_mal' in concepts:
        concept_guidance = "\n• Ayıplı malda tüketicinin seçeneklerini (değiştirme, onarım, iade) açıkla"

    system_prompt = f"""Sen Türk Tüketici Hukuku uzmanısın.

KURALLAR:
1. SADECE verilen kaynaklardaki bilgileri kullan
2. Net, anlaşılır ve yapılandırılmış yanıt ver
3. İlk cümlede soruya doğrudan yanıt ver
4. Kaynak numaralarını belirt (örn: [Kaynak 1])
5. Yasal dayanakları (madde numarası) belirt
6. Pratik öneriler ver{contract_guidance}{concept_guidance}

FORMAT:
• Ana yanıt (2-3 cümle)
• Yasal dayanak (1 cümle)
• Pratik öneri (1 cümle - varsa)
"""

    user_prompt = f"""KAYNAKLAR:

{context_block}

SORU: {query}

Yukarıdaki kaynaklara göre yanıt ver."""

    return system_prompt, user_prompt, source_details

# ============================================================================
# LLM LOADER
# ============================================================================

def load_llm_from_huggingface(model_id: str, max_seq_length: int):
    """Hugging Face'den model yükle"""
    print(f"\n📄 Hugging Face'den model yükleniyor: {model_id}...")

    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=max_seq_length,
            dtype=None,
            load_in_4bit=True,
        )

        FastLanguageModel.for_inference(model)
        print("✅ Model başarıyla yüklendi!")
        return model, tokenizer

    except Exception as e:
        print(f"❌ Model yükleme hatası: {e}")
        return None, None

# ============================================================================
# RAG CHAT FUNCTION
# ============================================================================

def gradio_chat_smart(message, history):
    """✅ DÜZELTİLMİŞ: Gerçekçi threshold'larla RAG chat"""
    if not llm_model:
        yield "❌ LLM yüklenemedi."
        return

    # Smart search
    search_results = vector_store.smart_search(query=message, n_results=5)

    if search_results['n_results'] == 0:
        yield "Üzgünüm, bu konuda yeterli kaliteli kaynak bulunamadı."
        return

    # Smart prompt
    system_prompt, user_prompt, source_details = create_smart_prompt(message, search_results)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Generate
    input_ids = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(DEVICE)

    streamer = TextIteratorStreamer(llm_tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        input_ids=input_ids,
        streamer=streamer,
        max_new_tokens=500,
        temperature=0.4,
        top_p=0.9,
        repetition_penalty=1.15,
        do_sample=True,
        pad_token_id=llm_tokenizer.eos_token_id,
    )

    thread = Thread(target=llm_model.generate, kwargs=generation_kwargs)
    thread.start()

    response = ""
    for new_text in streamer:
        response += new_text
        if len(response) > 50:
            yield response

    thread.join()

    # Kaynakları ekle
    if source_details:
        source_block = "\n\n" + "─"*50 + "\n**📚 Kaynaklar:**\n" + "\n".join([f"• {s}" for s in source_details])
        yield response + source_block

# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("\n" + "="*60)
print("🚀 DÜZELTİLMİŞ HUKUK PUSULASI - SEMANTİK RAG")
print("="*60)

llm_model, llm_tokenizer = load_llm_from_huggingface(HF_MODEL_ID, MAX_SEQ_LENGTH)

vector_store = SmartVectorStore(
    CHROMADB_PERSIST_DIR,
    CHROMADB_COLLECTION_NAME,
    EMBEDDING_MODEL_NAME
)

if llm_model and vector_store:
    print("\n✅ Sistem hazır! Gradio başlatılıyor...\n")

    demo = gr.ChatInterface(
        fn=gradio_chat_smart,
        examples=[
            "Online alışverişte cayma hakkım var mı?",
            "Telefon aldım ama kırık çıktı ne yapmalıyım?",
            "Ön ödemeli konut için cayma süresi ne kadar?",
            "Kusurlu mal aldım iade edebilir miyim?",
            "Kapıdan satış yapan firmadan aldım vazgeçebilir miyim?",
        ],
        title="⚖️ Akıllı Hukuk Pusulası - Düzeltilmiş RAG",
        description="""
**🎯 DÜZELTME: Gerçekçi Distance Threshold'lar ile Çalışır!**

**Ana Sorun:**
❌ Eski threshold'lar: 0.3, 0.5, 0.7 (ChromaDB L2 distance için ÇOK DÜŞÜK!)
✅ Yeni threshold'lar: 200, 350, 500 (GERÇEKÇİ!)

**Özellikler:**
✅ Semantic understanding (SentenceTransformers)
✅ Adaptive search strategy
✅ Doğru distance filtering (200-500 arası)
✅ Context-aware prompt engineering
✅ Quality-based result filtering

**Neden Çalışıyor:**
ChromaDB **L2 (squared Euclidean) distance** kullanır:
• Çok iyi eşleşme: 0-200
• İyi eşleşme: 200-350
• Kabul edilebilir: 350-500
• Zayıf: 500+

**Test edin:** "kırık telefon geldi", "aklımı değiştirdim" gibi doğal cümleler!
        """,
        theme="soft",
    )

    demo.launch(share=True, debug=True)
else:
    print("❌ Sistem başlatılamadı. Model veya vector store yüklenemedi.")



🚀 DÜZELTİLMİŞ HUKUK PUSULASI - SEMANTİK RAG

📄 Hugging Face'den model yükleniyor: beyzasn/hukuk-pusulasi-llm-v1...
==((====))==  Unsloth 2025.12.1: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Model başarıyla yüklendi!
✅ ChromaDB: 3158 doküman yüklendi

✅ Sistem hazır! Gradio başlatılıyor...



/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://aa2f7dcd424c0d0a91.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



🔍 Sorgu: 'Online alışverişte cayma hakkım var mı?'
📊 Semantic Analiz:
   • Sözleşme tipi: Belirsiz
   • Yasal konseptler: ['cayma_hakki (0.75)']
🎯 Zenginleştirilmiş: Online alışverişte cayma hakkım var mı? cayma hakkı iade vazgeçme...

📋 Strateji: Semantic=0.7, Filtered=0.1

2️⃣ Semantic search...
   ✅ 15 sonuç

✅ Toplam 15 kaliteli sonuç
📊 Mesafeler: ['262.5', '274.6', '285.4']

🔍 Sorgu: 'Ön ödemeli konut için cayma süresi ne kadar?'
📊 Semantic Analiz:
   • Sözleşme tipi: konut
   • Yasal konseptler: ['cayma_hakki (0.46)']
🎯 Zenginleştirilmiş: Ön ödemeli konut için cayma süresi ne kadar? ön ödemeli konut finansmanı...

📋 Strateji: Semantic=0.3, Filtered=0.6

1️⃣ Filtered search...

2️⃣ Semantic search...
   ✅ 10 sonuç

✅ Toplam 8 kaliteli sonuç
📊 Mesafeler: ['264.1', '267.2', '287.0']
